In [ ]:
! pip install lxml

# Premier League Data Collection

This notebook scrapes **Premier League match and shooting statistics** from [FBRef](https://fbref.com/) for the 2018–2022 seasons.  
The scraped dataset is later used to train machine learning models that predict match outcomes.  

---

## Objectives
- Collect match data from 2018–2022 seasons.  
- Merge "Scores & Fixtures" with "Shooting" statistics per team.  
- Save all matches into a single CSV (`matches.csv`) for downstream ML tasks.  

---

## 1. Setup & Imports
We install and import the required libraries:  
- **pandas** for data handling  
- **requests** and **BeautifulSoup** for web scraping  
- **time** for delays (respecting FBRef’s rate limits)  


In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time

## 2. Define Seasons & URLs

We will scrape **5 seasons (2018–2022)**.  
Each season corresponds to a Premier League statistics page on FBRef.  


In [ ]:
# Create a list with all the seasons that we are going to scrape in decending order
seasons = list(range(2022,2017, -1))
seasons

In [ ]:
# initiate an array that will eventually be a set of dataframes each containing matches played
# by a team from all seasons
team_seperated_matches = []
 

In [ ]:
league_table_urls =[
    "https://fbref.com/en/comps/9/Premier-League-Stats",
    "https://fbref.com/en/comps/9/2021-2022/2021-2022-Premier-League-Stats",
    "https://fbref.com/en/comps/9/2020-2021/2020-2021-Premier-League-Stats",
    "https://fbref.com/en/comps/9/2019-2020/2019-2020-Premier-League-Stats",
    "https://fbref.com/en/comps/9/2018-2019/2018-2019-Premier-League-Stats"]

## 3. Scrape Club Data

For each season:
1. Request the league table page.  
2. Extract links to each club’s page.  
3. From each club page:  
   - Collect "Scores & Fixtures".  
   - Collect "Shooting" statistics.  
   - Merge on the **Date** column.  

We add `Season` and `Team` metadata so matches can be identified later.  


In [ ]:
for season in seasons:
    time.sleep(5) # Some websites such as FBref allow web scraping but not super fast web scraping so we must use sleep 
    seasonNo = seasons.index(season)
    print(seasonNo)
   
    season_link = league_table_urls[seasonNo]
    print(season_link)

    data = requests.get(season_link)
    soup = BeautifulSoup(data.text)
    league_table = soup.select('table.stats_table')[0]
    
    urls = [url.get("href") for url in league_table.find_all('a')]
    urls = [url for url in urls if '/squads/' in url]
    club_urls = [f"https://fbref.com{url}" for url in urls]
    
    for club_url in club_urls:
        time.sleep(10) # Some websites such as FBref allow web scraping but not super fast web scraping so we must use sleep

        club_name = club_url.split("/")[-1].replace("-Stats", "").replace("-", " ")
        
        data = requests.get(club_url)
        matches = pd.read_html(data.text, match="Scores & Fixtures")[0]
        
        soup = BeautifulSoup(data.text)
        urls_ = [url.get("href") for url in soup.find_all('a')]

        urls_ = [url for url in urls_ if url and 'all_comps/shooting/' in url]

        data = requests.get(f"https://fbref.com{urls_[0]}")

        shooting = pd.read_html(data.text, match="Shooting")[0]
        shooting.columns = shooting.columns.droplevel()
        
        
        # For some teams the shooting stats aren't available, pandas will throw a value error, so to avoid that we 
        # just skip those teams
        try:
            team_data = matches.merge(shooting[["Date", "Sh", "SoT", "Dist", "FK", "PK", "PKatt"]], on="Date")
        except ValueError:
            continue
        except KeyError:
            continue
            
        team_data = team_data[team_data["Comp"] == "Premier League"] # We only want premier league data as the given 
                                                                     # data table has multiple competitions
        
        team_data["Season"] = seasons[seasonNo]
        team_data["Team"] = club_name
        team_seperated_matches.append(team_data)
                


## 4. Combine and Save Data

We concatenate all team DataFrames into a single dataset.  
- ~200 matches per season × 5 seasons.  
- Features include: Date, Team, Opponent, Sh, SoT, Dist, FK, PK, PKatt.  
 

The dataset is exported to `matches.csv` for later modelling.


In [ ]:
matches_df = pd.concat(team_seperated_matches)

In [ ]:
matches_df.describe()

## 5. Quick Checks
We run a few  checks:
- Inspect the shape of the dataset  
- Preview some rows  
- Verify that merged features look consistent

In [ ]:
matches_df

In [ ]:
testing_rows = matches_df[matches_df['Date'] == '2018-08-25'].head(10)
testing_rows

In [ ]:
matches_df.to_csv("matches.csv")

## 6. Next Steps

This dataset will be used in the **Premier League Prediction** notebook:  
- Feature engineering (venue, time, rolling averages, etc.)  
- Model training (Random Forest, etc.)  
- Performance evaluation in predicting match outcomes.  
